# 50 · Pooled paper analysis (tdd_paper)

Thin notebook over the `tdd_paper` package: builds the pooled 120-run dataset across the ten wave-1 stories,
runs the integrity gate, the stratified contrasts (Cliff's delta with bootstrap intervals, van Elteren test,
Holm over the two declared families), the TOST equivalence tests, pass@k / pass^k reliability, the composite
ranking and the sensitivity sets, then writes every number, table and figure to `OUT_DIR`.

Provenance rule: only tool-measured and git-derived values are scored; agent self-reports are tagged and
appear only in the provenance family. Run headless with
`.venv/bin/python -m papermill 50_pooled_analysis.ipynb _executed/50.ipynb -p OUT_DIR _build/paper`.


In [ ]:
OUT_DIR = "_build/paper"
SEED = 20261003
N_BOOT = 5000
N_PERM = 20000
WITH_STABILITY = True


In [ ]:
from pathlib import Path
import pandas as pd
from tdd_paper import dataset as ds, analysis as an, checks as K, figures as F, paper_tables as PT, theme as T
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
out = Path(OUT_DIR)
bundle = ds.build_runs(progress=lambda s: print("built", s))
ds.write_dataset(bundle, out)
runs = bundle["runs"]
runs.shape


## Integrity gate


In [ ]:
checks = K.run_checks(bundle)
print(K.report(checks))
assert not any(not c.ok and c.required for c in checks), "integrity gate failed"


## Descriptives and contrasts


In [ ]:
res = an.run_all(bundle, n_boot=N_BOOT, n_perm=N_PERM, seed=SEED, with_stability=WITH_STABILITY,
                 progress=lambda s: print("stats:", s))
an.write_frames(res, out); res.numbers.write(out)
eff = res.frames["effects"]
eff[eff.family != "exploratory"][["metric", "family", "median_a", "median_b", "delta", "ci_low", "ci_high", "p_used", "p_holm", "reject", "stories_kimi_higher", "stories_qwen_higher", "i2", "robustness"]].round(3)


In [ ]:
res.frames["tost"][["metric", "band", "estimate", "ci_low", "ci_high", "equivalent", "smallest_equivalent_band"]].round(3)


## Reliability, ranking and sensitivity


In [ ]:
display(res.frames["rank_summary"])
display(res.frames["robustness"])
res.frames["passk"].query("k in (1, 3, 6)").pivot_table(index=["predicate", "group"], columns=["estimator", "k"], values="mean").round(2)


## Figures, tables and manifest


In [ ]:
import hashlib
sha = hashlib.sha256((out / "data" / "runs.csv").read_bytes()).hexdigest()
ctx = F.FigureContext(runs=runs, executions=bundle["executions"], types=bundle["types"], cells=bundle["cells"],
                      stories=bundle["stories"], frames=res.frames, out_dir=out, seed=SEED, runs_sha256=sha)
figs = F.make_figures(ctx, progress=lambda slug, err: print(slug, "ERROR" if err else "ok"))
tables = PT.make_tables(bundle, res.frames, out)
K.write_manifest(out, extra={"seed": SEED, "n_boot": N_BOOT, "n_perm": N_PERM, "source": "50_pooled_analysis.ipynb"})
print(len(res.numbers.entries), "numbers;", sum('error' not in v for v in figs.values()), "figures;", len(tables), "tables")


In [ ]:
from IPython.display import Image, display
for slug in ("fig_forest_delta", "fig_convergence", "fig_red_first"):
    display(Image(filename=str(out / "figures" / f"{slug}.png"), width=900))
